# Model comparison: revised pipeline vs. Sugiyama et al. 2023

**Purpose.** Evaluate the revised likelihood (`scripts/hsc_lens.py`) at the *published* MAP
parameters and compare, probe by probe, against the literature best-fit theory vector and the
published $\ln\mathcal{L}$. This is the Phase-1 validation gate: all three fixes (PSF term,
pair-weighted $\langle\Sigma_{\rm cr}^{-1}\rangle$, linear Kaiser ratio) should close the
discrepancies seen in the earlier `mine/literature` ratio plot.

**Prerequisites** (paths relative to repo root):
- `data/dataset_hsc_y3.fits` — with the **corrected** lens n(z) (CMASS bins at their true z)
- `data/dataset/psf_pp_pq_qq_used.dat`, `sumwlssigcritinvPz_z[0-2].dat`, `photoz_bin.dat`
- `data/chain_equal_weights.dat` + `data/param_names.dat` (published chain)
- optional: `data/lit_bestfit_signal.txt` (released best-fit theory vector, masked, length 74)

**Pass criteria**
1. every point of `mine/lit` within the ±1% band (shaded) — or within 0.1σ pulls;
2. $\chi^2$ at the published MAP within ~0.5 of $-2\ln\mathcal{L}$ from the chain.

In [1]:
import os, sys, copy
import numpy as np
import matplotlib.pyplot as plt
import yaml

REPO = os.path.abspath('..')          # notebook lives in notebooks/
sys.path.insert(0, os.path.join(REPO, 'scripts'))
sys.path.insert(0, os.path.join(REPO, 'theory'))

DATA = os.path.join(REPO, 'data')
CHAIN_FILE  = os.path.join(DATA, 'chain_equal_weights.dat')
PNAMES_FILE = os.path.join(DATA, 'param_names.dat')
LIT_SIGNAL  = os.path.join(DATA, 'lit_bestfit_signal.txt')   # optional

plt.rcParams.update({'figure.dpi': 110, 'font.size': 11})

## 1. Build the likelihood through cobaya

We instantiate via `get_model` on the ΛCDM config so priors/params match exactly what an MCMC
would use, then grab the likelihood object to access its internals
(`_build_cosmology`, `theory_vector`, masks, covariance).

In [3]:
from cobaya.model import get_model

with open('/home/weichen/cosmo_practice/test_cobaya/test_cobaya/lcdm_hsc_only.yaml') as f:
    info = yaml.safe_load(f)
info.pop('output', None)
info.pop('sampler', None)
info['likelihood']['hsc_lens.HSC_Lens']['python_path'] = os.path.join(REPO, 'scripts')
info['likelihood']['hsc_lens.HSC_Lens']['data_folder'] = DATA

model = get_model(info)
like = list(model.likelihood.values())[0]
print('data vector length:', len(like.data_vector))
print('blocks: ds=%d xip=%d xim=%d wp=%d' % (like.ds_cut.sum(), like.xip_cut.sum(),
                                             like.xim_cut.sum(), like.wp_cut.sum()))

ComponentNotFoundError: 'hsc_lens.HSC_Lens' could not be found. Tried loading from /home/weichen/cosmo_practice/test_cobaya/scripts

## 2. Published MAP point

Load the equal-weight chain and its parameter names, take the maximum-posterior row, and map
their names onto ours. **Edit `NAME_MAP` after inspecting the printed names** — cosmology
naming in `param_names.dat` must be verified once by eye (the nuisance names below were
confirmed earlier from the released chain).

In [ ]:
pnames = [l.strip() for l in open(PNAMES_FILE) if l.strip()]
chain = np.loadtxt(CHAIN_FILE)
print('chain shape:', chain.shape, ' n_names:', len(pnames))
for i, n in enumerate(pnames):
    print(f'{i:3d}  {n}')

In [ ]:
# ---- EDIT AFTER INSPECTING THE PRINTOUT ABOVE ----------------------------
# our param  ->  candidate names in param_names.dat (first match wins)
NAME_MAP = {
    'logA':      ['ln10p10As', 'lnAs', 'ln10^{10}As', 'logAs'],
    'ombh2':     ['omega_b', 'ombh2', 'Omega_b_h2'],
    'omch2':     ['omega_c', 'omch2', 'Omega_c_h2'],
    'H0':        ['H0', 'h0'],
    'ns':        ['ns', 'n_s'],
    'b1':        ['b1_0', 'b1z0', 'b1_L0'],
    'b2':        ['b1_1', 'b1z1', 'b1_L1'],
    'b3':        ['b1_2', 'b1z2', 'b1_L2'],
    'AIA':       ['AIA', 'A_IA'],
    'dm_0':      ['dm_0', 'dm'],
    'dpz_0':     ['dpz_0', 'dpz'],
    'alphapsf':  ['alphapsf'],
    'betapsf':   ['betapsf'],
    'alphamag_1':['alphamag_0'],
    'alphamag_2':['alphamag_1'],
    'alphamag_3':['alphamag_2'],
}
# column with the posterior/likelihood (check header/README of the chain!)
LNP_CANDIDATES = ['lnpost', 'lnlike', 'lnP', 'loglike', 'weight']

def find_col(cands):
    for c in cands:
        if c in pnames:
            return pnames.index(c)
    return None

lnp_col = find_col(LNP_CANDIDATES)
if lnp_col is None:
    print('!! no lnpost/lnlike column found by name; assuming LAST column is lnlike')
    lnp_col = chain.shape[1] - 1
imap = int(np.argmax(chain[:, lnp_col]))
print('MAP row:', imap, ' lnP =', chain[imap, lnp_col])

map_params = {}
for ours, cands in NAME_MAP.items():
    col = find_col(cands)
    if col is None:
        raise KeyError(f'no column found for {ours}: tried {cands} -- edit NAME_MAP')
    map_params[ours] = float(chain[imap, col])
map_params

## 3. Evaluate our model at the MAP

`model.logposterior` fills derived params and returns the log-likelihood; we also pull the
theory vector directly. Then re-evaluate with each fix disabled to attribute residuals.

In [ ]:
def theory_at(params, *, psf=True, pairweight=True, kaiser=True):
    """Theory vector at `params` with individual fixes toggled."""
    import test_cobaya.scripts.hsc_lens as HL
    like_ = like
    # toggle PSF
    psf_save = like_.psf
    if not psf: like_.psf = np.zeros_like(psf_save)
    # toggle pair-weighted Sigma_cr correction
    mc_save = like_.meascorr
    if not pairweight: like_.meascorr = None
    # toggle Kaiser (patch the ratio to unity)
    kr_save = HL._kaiser_ratio
    if not kaiser:
        HL._kaiser_ratio = lambda r, xi, beta, pimax: (lambda rp: np.ones_like(np.atleast_1d(rp)))
    try:
        pv = dict(params)
        pv['As'] = 1e-10 * np.exp(pv.pop('logA'))
        base, cosmo, gr, om = like_._build_cosmology(pv)
        vec = like_.theory_vector(base, cosmo, gr, om, pv)
    finally:
        like_.psf = psf_save
        like_.meascorr = mc_save
        HL._kaiser_ratio = kr_save
    return vec

vec_full     = theory_at(map_params)
vec_nopsf    = theory_at(map_params, psf=False)
vec_stacked  = theory_at(map_params, pairweight=False)
vec_nokaiser = theory_at(map_params, kaiser=False)

d = like.data_vector - vec_full
chi2 = float(d @ like.inv_cov @ d)
print(f'chi2(full model at published MAP) = {chi2:.2f}   (N_data = {len(d)})')
print(f'published -2 lnL at MAP           = {-2*chain[imap, lnp_col]:.2f}  <-- verify column meaning')

## 4. Ratio to the literature best-fit vector

If the released best-fit theory vector is available, reproduce the diagnostic ratio plot —
now with the before/after variants overlaid, so each fix is visibly responsible for closing
its block. (If the file is absent, skip to the pulls plot in §5, which needs only the data.)

In [ ]:
blocks = [('ds', like.ds_cut.sum(), 'C0'), ('xip', like.xip_cut.sum(), 'C1'),
          ('xim', like.xim_cut.sum(), 'C1'), ('wp', like.wp_cut.sum(), 'C2')]
edges = np.cumsum([0] + [n for _, n, _ in blocks])

if os.path.exists(LIT_SIGNAL):
    lit = np.loadtxt(LIT_SIGNAL)
    assert lit.size == len(vec_full), f'lit vector {lit.size} vs ours {len(vec_full)}'

    fig, ax = plt.subplots(figsize=(11, 4))
    ax.axhspan(0.99, 1.01, color='C0', alpha=0.12, lw=0)
    ax.axhline(1.0, color='k', lw=0.8)
    x = np.arange(len(vec_full))
    for (name, n, c), lo, hi in zip(blocks, edges[:-1], edges[1:]):
        ax.plot(x[lo:hi], (vec_full/lit)[lo:hi], 'o-', ms=4, color=c, label=None)
    # dashed: fixes disabled (should reproduce the OLD discrepancies)
    ax.plot(x, vec_nopsf/lit,    ':', color='C1', alpha=.6, label='PSF term off')
    ax.plot(x, vec_stacked/lit,  ':', color='C0', alpha=.6, label=r'stacked p(z) $\Sigma_c$')
    ax.plot(x, vec_nokaiser/lit, ':', color='C2', alpha=.6, label='Kaiser off')
    for e in edges[1:-1]:
        ax.axvline(e-.5, color='gray', lw=.5, alpha=.5)
    for (name, n, c), lo in zip(blocks, edges[:-1]):
        ax.text(lo + n/2, ax.get_ylim()[1], name, ha='center', va='bottom', color=c)
    ax.set_xlabel('data-vector index'); ax.set_ylabel('mine / literature')
    ax.legend(loc='lower right', fontsize=9); plt.tight_layout(); plt.show()
else:
    print('lit_bestfit_signal.txt not found -- skipping; use the pulls plot below.')

## 5. Pulls against the data

Independent of any released theory vector: residuals of our MAP prediction against the *data*
in units of the diagonal errors. At the true MAP these should scatter like the published fit
does (reduced $\chi^2 \approx$ theirs), with **no coherent per-block offset**.

In [ ]:
sig = 1/np.sqrt(np.diag(like.inv_cov))   # note: includes Hartlap; fine for pulls
pull = (like.data_vector - vec_full) / sig

fig, ax = plt.subplots(figsize=(11, 3.2))
x = np.arange(len(pull))
for (name, n, c), lo, hi in zip(blocks, edges[:-1], edges[1:]):
    ax.plot(x[lo:hi], pull[lo:hi], 'o-', ms=4, color=c)
    ax.text(lo + n/2, 2.6, name, ha='center', color=c)
ax.axhline(0, color='k', lw=.8); ax.set_ylim(-3, 3)
for e in edges[1:-1]: ax.axvline(e-.5, color='gray', lw=.5, alpha=.5)
ax.set_xlabel('data-vector index'); ax.set_ylabel(r'(data $-$ model)/$\sigma$')
plt.tight_layout(); plt.show()

for (name, n, c), lo, hi in zip(blocks, edges[:-1], edges[1:]):
    print(f'{name:4s}: mean pull = {pull[lo:hi].mean():+.3f}, chi2/n = '
          f'{np.sum(pull[lo:hi]**2)/n:.2f}')

## 6. Attribution table

$\Delta\chi^2$ contribution of each fix, evaluated at the published MAP. Large positive
numbers = removing the fix badly degrades the fit = the fix matters.

In [ ]:
def chi2_of(vec):
    r = like.data_vector - vec
    return float(r @ like.inv_cov @ r)

rows = [('full model (all fixes)', vec_full),
        ('PSF term disabled',      vec_nopsf),
        ('stacked-p(z) Sigma_cr',  vec_stacked),
        ('Kaiser ratio disabled',  vec_nokaiser)]
print(f'{"variant":28s} {"chi2":>9s} {"dchi2 vs full":>14s}')
c0 = chi2_of(vec_full)
for name, v in rows:
    c = chi2_of(v)
    print(f'{name:28s} {c:9.2f} {c-c0:14.2f}')

## 7. Troubleshooting map

| symptom (which block off) | first thing to check |
|---|---|
| $\xi_+$ low, worsening with $\theta$; $\xi_-$ fine | PSF file/columns (pp+, pp−, pq+, pq−, qq+, qq−); term added *after* $(1+\Delta m)^2$ |
| $\Delta\Sigma$ per-bin steps growing with lens $z$ | `sumwlssigcritinvPz` loading; sign of $z_s - \Delta z$; `photoz_bin.dat` grid |
| $w_p$ droop growing with $R$ | Kaiser ratio built from **linear** $\xi$; $\beta=f/b$; $\Pi_{\rm max}/E_{\rm fac}$ |
| overall few-% amplitude everywhere | $A_s$ vs $\sigma_8$ handling; $m_\nu$ in CCL; halofit flavor vs `analysis_config.yaml` |
| sub-% sawtooth within bins | bin-averaging (`binave`, $R\,dR$ weight) vs bin-center evaluation |
| $\chi^2$ off by O(1) with perfect vectors | Hartlap $N_{\rm sim}=1404$; mask/cut ordering vs published covariance |

When every point sits inside ±1% and the MAP $\chi^2$ matches, tag the commit
(`v0-lcdm-reproduction`) and record the numbers in the Phase-1 note.